# CS 577: Principles and Techniques of Data Science
## Phase 4 (Group): Build Your Model and See If It Holds Up
**Fall 2026 | San Diego State University | 100 points**

So in Phase 3 you made your case for a target and what kind of model fits your data. Now we actually build it. You'll start with a baseline, see if a simple model can beat it, add more features, play with a few settings, try a random forest, and then at the very end find out which model really did the best.

Some of you changed your questions or redid your EDA, which honestly is expected, so Part 0 is where you tell us what changed and catch up.

**How this notebook works.** Every part has one markdown cell with the instructions and one code cell right under it. Write your answers in the markdown cell where it says **Your answer**, and fill in every `TODO` in the code. Please don't delete or move parts around, but if you need extra cells inside a part that's fine.

| Part | What you do | Points |
|---|---|---|
| 0 | Pick your track, tell us what changed, show us your data | 15 |
| 1 | Translate your plots for the model | 10 |
| 2 | Seal the future | 5 |
| 3 | Baseline | 10 |
| 4 | Just one feature | 10 |
| 5 | Now add all your features | 10 |
| 6 | Turn the dial | 15 |
| 7 | Random forest | 10 |
| 8 | Open the envelope | 10 |
| 9 | README, contributions, and AI use | 5 |
| | **Total** | **100** |

**A few rules for the whole notebook, please actually read these:**
1. **Don't print your whole DataFrame.** No typing `df` by itself and nothing that shows more than 20 rows, because it crashes Canvas and it crashes grading. Use `df.shape`, `df.dtypes`, and `df.sample(10)` instead.
2. **Leave `future` alone.** In Part 2 you set aside some rows called `future`. Don't plot them, describe them, or score them until Part 8.
3. **It has to run.** Your notebook needs to run top to bottom with Kernel, Restart and Run All, using only your sample file.

If you're a team of one, just read "both members" as "you" and submit to the group assignment.

**What to submit:** this notebook named `GroupNumber_FirstName1_FirstName2_Phase4.ipynb`, an HTML export of it, and your `phase4_sample.csv`.

The rubric is at the very bottom so you know exactly what we're looking for.

---
### Setup
Run this cell first and don't change anything in it. It loads everything you need, plus four helpers we use the whole way through:

* `error(y, y_hat)` tells you how wrong your predictions are. Track A gets RMSE, so lower is better. Track B gets accuracy, so higher is better.
* `scaled(frame, cols)` puts your columns on the same scale using the averages from `past` only.
* `save(name, model, cols)` saves your model and prints how it did on `past`.
* `open_envelope()` checks every model you saved on `future`. You only use this once, in Part 8.

In [1]:
# Setup: run this first and please don't change anything in here.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

pd.set_option("display.max_rows", 20)   # keeps every output small
SEED = 577
scoreboard = []                         # every model you save lands here for Part 8


def error(y, y_hat):
    """Track A: RMSE, lower is better. Track B: accuracy, higher is better."""
    y, y_hat = np.asarray(y), np.asarray(y_hat)
    if TRACK == "A":
        return np.sqrt(((y - y_hat) ** 2).mean())
    return (y == y_hat).mean()


def scaled(frame, cols):
    """Put columns on the same scale, using the averages from past only."""
    spread = past[cols].std().replace(0, 1)
    return (frame[cols] - past[cols].mean()) / spread


def save(name, model, cols, scale=False):
    """Record a fitted model and print its score on past."""
    X = scaled(past, cols) if scale else past[cols]
    score = error(past[TARGET], model.predict(X))
    scoreboard.append({"name": name, "model": model, "cols": list(cols),
                       "scale": scale, "past": score})
    print(f"{name}: {score:.4f} on past")


class Baseline:
    """A 'model' that predicts the same value for every row."""
    def __init__(self, value):
        self.value = value
    def predict(self, X):
        return np.array([self.value] * len(X))


def open_envelope():
    """Score every saved model on future. Use this once, in Part 8."""
    rows = []
    for m in scoreboard:
        X = scaled(future, m["cols"]) if m["scale"] else future[m["cols"]]
        rows.append({"model": m["name"],
                     "past": round(m["past"], 4),
                     "future": round(error(future[TARGET], m["model"].predict(X)), 4)})
    table = pd.DataFrame(rows)
    table["change"] = (table["future"] - table["past"]).round(4)
    return table

print("Setup done.")

Setup done.


## Part 0: Pick Your Track, Tell Us What Changed, Show Us Your Data (15 points)

**0A. Pick your track.** This phase works for both kinds of questions, so first figure out which one you are. Go back to your target from Phase 3 Part 4A. If it's a number where math actually makes sense (price, minutes, score), you're Track A. If it's a category (yes or no, a tier, a genre), you're Track B.

|                   | Track A: Predict a Number           | Track B: Predict a Category          |
|-------------------|-------------------------------------|--------------------------------------|
| Target            | Quantitative: price, minutes, score | Qualitative: yes or no, tier, genre  |
| Baseline          | The mean, for every row             | The most common class, for every row |
| How wrong are we? | RMSE (lower is better)              | Accuracy (higher is better)          |
| Simple model      | Linear regression                   | Logistic regression                  |
| Part 6 dial       | Ridge or Lasso strength             | Decision tree depth                  |
| Part 7            | Random forest regressor             | Random forest classifier             |

Careful with numbers that are really labels, like ZIP codes or survey codes 1 to 5, those are Track B. And if you turned a rating into "4 or above" vs "below 4", that's Track B too. Once you start Part 3 your track is locked, so pick carefully.

**0B. What changed?** A lot of you changed your question or redid your EDA and that's okay, just tell us. Answer yes or no for each one and explain any yes.
1. Did your research question change? If yes, write the Phase 3 version and the new one so we can see the difference.
2. Did your target or your track change?
3. Did your dataset change? More cleaning in code is fine, you don't need to ask for that. Switching to a whole different dataset needs my permission first, so tell us when you got it.

**0C. Show us your data.** Give us a link to your full dataset along with its shape and file size. Then in the code cell make a sample of at most 20,000 rows. Run that part once, save it, and comment it back out. Load the sample into pandas and show us the data using only `shape`, `dtypes`, and `sample(10)`. Please don't print the whole thing, it crashes the system. If your target is a category and one class is really rare, sample inside each class instead so it doesn't disappear: `full.groupby(TARGET).sample(frac=0.10, random_state=SEED)`.

**0D. Catch up.**
* **If your question, target, or dataset changed,** your Phase 3 model case doesn't really fit anymore, so redo the important parts here. Add plots to the code cell showing how your target is distributed, plus three plots of your target against features (one split by a category, one on a log scale). Then in 3 to 5 sentences tell us which features leak and whether you're going linear or forest.
* **If nothing changed,** tell us in 3 to 5 sentences which of your Phase 3 figures show your model case still holds, and what you did with your Phase 3 feedback. "It was fine" gets zero, I want to see that you actually checked.

**In the code cell:** set `TRACK` and `TARGET`. Uncomment the sample lines, change the file name if yours is different, run it once, then comment them back out. If any column needs `parse_dates` or `dtype`, add it to the `read_csv` line. Only add plots at the bottom if something changed (0D).

**Your answer:**

* Track: *"Our target is **NA_Sales**, which is **quantitative**, so we are on Track **A**."*
* One row in my dataset represents **one game title on one platform**.
* Our research question right now: **Can a game's genre and platform be used to predict its North American sales, among VGChartz-tracked titles released through 2020 on platforms with at least 100 tracked titles?**
* 0B:
    1. No, our research question did not change from phase 3.
    2. No, the target (NA_Sales) and track (A) did not change.
    3. No, the dataset did not change. We are still using phase3_working.csv.
* Full original dataset:
    * Link: https://www.kaggle.com/datasets/lamskdna/video-games-sales/data
    * Shape: 64,016 rows x 12 columns.
    * File size: 3.27 MB.
        * After two rounds of cleaning the dataset, our present working dataset is: *phase3_working.csv: 12,031 rows x 8 columns.*

* 0D: **Phase 3 feedback:** The notebook read Phase 2 dataset from another folder because of how we organized our project, so we will meet with the professor to demonstrate it runs so it can be graded. Our target, dataset, and model case remain the same as in Phase 3. Figure M1 still supports our case, because the platform lines cross: X360 has the highest median in five of the six genres, but Wii overtakes it in Misc and falls to last in Shooter and Racing. Figure M2 still shows the threshold in critic score, where median sales stay nearly flat until a score of about 8 and then rise sharply. In Phase 3 we left Critic_Score out of the model because only 29.8% of titles had one; in Phase 4 we include it by filling the missing scores with 0 and adding a `Has_Critic_Score` flag so the model can treat unscored games separately, and the column does not leak because reviews come out around launch, before most sales accumulate.

In [2]:
# Part 0: pick your track, then load your sample.
TRACK = "A"     # "A" (predict a number) or "B" (predict a category)
TARGET = "NA_Sales"    # your target column from Phase 3 Part 4A

# Run these lines just ONCE to make your sample, then comment them back out.
# full = pd.read_csv("phase3_working.csv")     # or phase2_working.csv
# print(full.shape)
# sample = full.sample(n=min(20000, len(full)), random_state=SEED)
# sample.to_csv("phase4_sample.csv", index=False)

df = pd.read_csv("phase4_sample.csv", parse_dates=["Release_Date"], date_format="%m/%d/%Y")   # add parse_dates=[...] or dtype={...} if your columns need it

assert TRACK in ("A", "B"), 'You need to set TRACK to "A" or "B" first'
assert TARGET in df.columns, "TARGET has to be the name of a column in your data, check the spelling"

# Show us your data with only these three, no printing the whole thing or it crashes:
print(f"The DataFrame shape is {df.shape}.")
print(f"The DataFrame Type is \n{df.dtypes}.")
display(df.sample(10, random_state=SEED))

# 0D: if your question, target, or dataset changed, put your catch-up plots below.
# If nothing changed, leave this part empty.

The DataFrame shape is (12031, 8).
The DataFrame Type is 
Title                      str
Platform                   str
Genre                      str
Publisher                  str
Developer                  str
Critic_Score           float64
NA_Sales               float64
Release_Date    datetime64[us]
dtype: object.


,Title,Platform,Genre,Publisher,Developer,Critic_Score,NA_Sales,Release_Date
5563,Spirits & Spells,GBA,Platform,DreamCatcher Interactive,Magic Pockets,NaN,0.01,2003-11-02
4850,NBA Hoopz,PS,Sports,Midway Games,Eurocom Entertainment Software,NaN,0.07,2001-02-12
3753,Sniper Elite III,XOne,Shooter,505 Games,Rebellion Developments,NaN,0.27,2014-07-01
1811,Captain America: Super Soldier,PS3,Action,Sega,Next Level Games,6.2,0.10,2011-07-19
6146,Castlevania Double Pack,GBA,Platform,Konami,KCET,9.0,0.07,2006-01-11
9098,OutRun 2006: Coast 2 Coast,PSP,Racing,Sega,Sumo Digital,NaN,0.03,2006-04-25
6033,Valhalla Knights,PSP,Role-Playing,Xseed Games,K2 LLC,5.2,0.15,2007-04-17
9125,Create,PC,Misc,Electronic Arts,EA Bright Light,6.0,0.04,2010-11-16
5745,Rise of Flight: Iron Cross Edition,PC,Simulation,Unknown,777 Studios,NaN,0.01,2010-07-07
5777,L.A. Noire,XOne,Adventure,Rockstar Games,Team Bondi,NaN,0.14,2017-11-14


## Part 1: Translate Your Plots for the Model (10 points)

You can see the patterns in your plots but your model can't, it only reads numbers one row at a time. So your job here is to make at least three new columns that hand the model what your plots showed you.

1. **A curve.** If one of your plots got straighter on a log scale, make a log version of that column.
2. **A category.** If splitting by a category showed groups acting differently, turn that category into 0 and 1 columns with `pd.get_dummies`.
3. **A combination.** Make one column out of two others that you think matters more than either one alone, like price per square foot or age from year built.

One rule here: every new column can only use values from its own row, and it can never use your target. If you're not sure, go back to your Phase 3 leakage list.

Then put every column you want the model to use in `FEATURES`, and just the ones you made in `NEW_COLUMNS`. Everything in `FEATURES` has to be a number, so any text columns need to go through `pd.get_dummies` first.

**In the code cell:** replace the three examples with your own columns (the column names in there are made up, so don't just uncomment them). Run the cell once, it prints all your column names, including the new 0 and 1 columns from `get_dummies`. Copy the exact names into `FEATURES` and `NEW_COLUMNS` and run it again. If you get an error, read the message, it tells you what to fix.

**Your answer:** For each new column, write one sentence on which figure gave you the idea.

1. **Curve: `Critic_Score_Sq`:** In Figure M2, median NA sales stayed nearly flat until a critic score of about 8 and then rose sharply, even on a log scale, so squaring the score gives the model a column that grows faster at high scores, matching that threshold; we used a square instead of a log because M2 showed that the log barely helped (r rose from 0.30 to 0.35), and a log column flattens at high values, which is the opposite of the upward bend.
2. **Category: `Genre_*` and `Platform_*` dummies:** Figures A3 and B3 (and the Hero Figure) showed that median NA sales differ by genre and especially by platform, so each genre and platform becomes its own 0/1 column.
3. **Combination: `X360_Shooter`:** Figure M1 showed the platform lines cross across genres, with X360 holding its biggest lead in Shooter (0.26M median vs. 0.09M for Wii), so this column marks the games that are both X360 and Shooter.

We also brought `Critic_Score` back as a feature by filling missing scores with 0 and adding `Has_Critic_Score` (1 if reviewed), so the 70% of titles without a score are not dropped, and added `Release_Year`, since the upgraded Figure B4 showed sales change by era. We treat these three columns as base features rather than plot transformations so they are in `FEATURES` but not `NEW_COLUMNS`. Additionally, nine rows that did not have a release date were dropped from the DataFrame.

In [3]:
# Part 1: translate your plots into columns.
# Each new column can only use its own row, and never the target.

# Critic score manipulation stated above.
df["Has_Critic_Score"] = df["Critic_Score"].notna().astype(int)
df["Critic_Score"] = df["Critic_Score"].fillna(0)
df["Release_Year"] = df["Release_Date"].dt.year

# These examples use made up column names. Replace them with YOUR columns.

# 1. A curve. Example (use np.log1p instead if the column has any zeros, np.log(0) breaks the model):
df["Critic_Score_Sq"] = df["Critic_Score"] ** 2

# 2. A category. Example:
df = pd.get_dummies(df, columns=["Genre", "Platform"], drop_first=True, dtype=int)

# 3. A combination. Example:
df["X360_Shooter"] = df["Platform_X360"] * df["Genre_Shooter"]

print(list(df.columns))   # copy your exact column names from here, including the new 0 and 1 columns

genre_cols = [c for c in df.columns if c.startswith("Genre_")]
platform_cols = [c for c in df.columns if c.startswith("Platform_")]

NEW_COLUMNS = ["Critic_Score_Sq", "X360_Shooter"] + genre_cols + platform_cols  # just the ones you made above (these go in FEATURES too)
FEATURES = ["Critic_Score", "Has_Critic_Score", "Release_Year"] + NEW_COLUMNS   # every column you want the model to use, numbers only

assert set(NEW_COLUMNS) <= set(FEATURES), "Your new columns need to be in FEATURES too"
assert TARGET not in FEATURES, "Your target can't be one of its own features, that's leakage"
missing = [c for c in FEATURES if c not in df.columns]
assert not missing, f"These columns don't exist in df, check the spelling: {missing}"
assert all(pd.api.types.is_numeric_dtype(df[c]) for c in FEATURES), "Every feature has to be a number, run text columns through pd.get_dummies first"

df = df.dropna(subset=FEATURES + [TARGET])   # models can't handle blanks, so we drop those rows
print(df.shape)
display(df[FEATURES].describe())

['Title', 'Publisher', 'Developer', 'Critic_Score', 'NA_Sales', 'Release_Date', 'Has_Critic_Score', 'Release_Year', 'Critic_Score_Sq', 'Genre_Action-Adventure', 'Genre_Adventure', 'Genre_Fighting', 'Genre_Misc', 'Genre_Platform', 'Genre_Puzzle', 'Genre_Racing', 'Genre_Role-Playing', 'Genre_Shooter', 'Genre_Simulation', 'Genre_Sports', 'Genre_Strategy', 'Platform_3DS', 'Platform_DS', 'Platform_GBA', 'Platform_GC', 'Platform_N64', 'Platform_NS', 'Platform_PC', 'Platform_PS', 'Platform_PS2', 'Platform_PS3', 'Platform_PS4', 'Platform_PSP', 'Platform_PSV', 'Platform_Wii', 'Platform_X360', 'Platform_XB', 'Platform_XOne', 'X360_Shooter']
(12022, 39)


,Critic_Score,Has_Critic_Score,Release_Year,Critic_Score_Sq,X360_Shooter,Genre_Action-Adventure,Genre_Adventure,Genre_Fighting,Genre_Misc,Genre_Platform,...,Platform_PS,Platform_PS2,Platform_PS3,Platform_PS4,Platform_PSP,Platform_PSV,Platform_Wii,Platform_X360,Platform_XB,Platform_XOne
count,12022.000000,12022.000000,12022.000000,12022.000000,12022.000000,12022.000000,12022.000000,12022.000000,12022.000000,12022.00000,...,12022.000000,12022.000000,12022.000000,12022.000000,12022.000000,12022.000000,12022.000000,12022.000000,12022.000000,12022.000000
mean,2.110564,0.298286,2007.198636,15.543385,0.014473,0.015638,0.065879,0.046914,0.097571,0.06355,...,0.076776,0.129429,0.077774,0.054317,0.046997,0.012477,0.089835,0.090501,0.067876,0.039261
std,3.330139,0.457525,5.837793,25.936957,0.119437,0.124076,0.248081,0.211463,0.296746,0.24396,...,0.266247,0.335689,0.267827,0.226652,0.211642,0.111007,0.285958,0.286910,0.251543,0.194224
min,0.000000,0.000000,1977.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,2003.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,2008.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,5.700000,1.000000,2011.000000,32.490000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,10.000000,1.000000,2020.000000,100.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


## Part 2: Seal the Future (5 points)

At some point your model is going to be used on rows that don't exist yet, so let's practice that. The code below takes 20% of your rows and puts them aside in `future`, and everything else goes in `past`.

From here until Part 8 every model only learns from `past`. Don't plot `future`, don't describe it, don't score it. Think of it like a sealed envelope that we open at the end.

**In the code cell:** nothing to fill in, just run it once.

**Your answer:** In one or two sentences, why do you think a score on rows the model has never seen might be more honest than a score on rows it learned from?

* We think a score on rows the model learned from can look better than it actually is, because the model may memorize the noise of those specific rows instead of the general pattern, which is overfitting. Rows that have yet to be seen give a more honest picture of how well the model will predict new games.

In [4]:
# Part 2: seal the future. Run this once and don't change it.
future = df.sample(frac=0.2, random_state=SEED)
past = df.drop(future.index)

print("past:  ", past.shape)
print("future:", future.shape)
# From here until Part 8 you only use past. Don't touch future.

past:   (9618, 39)
future: (2404, 39)


## Part 3: Baseline (10 points)

The baseline is the laziest guess possible, and every model after this has to beat it. You already did this in Phase 3, now we just do it on `past`.

* **Track A:** predict the mean of your target for every row.
* **Track B:** predict the most common class for every row.

**In the code cell:** replace `None` with one line of pandas that computes `baseline_value` from `past`. For Track B, `value_counts()` might help. The code then saves it as your first model.

**Your answer:**
1. What's your baseline guess, in your target's units or class name?
    * Phase 3 baseline prediction: 0.270M (270,000 units per game)
    * Baseline on `past`: 0.273M (273,000 units per game). This baseline is slightly higher by chance because the mean comes from a random 80% of the rows, rather than all of them.
2. What does its score actually mean, in plain words?
    * Track A: "On average our guess is off by about 0.505 million units." Phase 3 reported a mean absolute error of 0.256M for nearly the same guess. Phase 4's RMSE is 0.505M, which is about twice as large. Since RMSE squares each miss, the few blockbuster games that sold several million copies pulled that number well above a typical miss, while MAE counts every miss equally.

In [5]:
# Part 3: baseline.
baseline_value = past[TARGET].mean()  # TODO: Track A, the mean of TARGET in past. Track B, the most common class in past.

assert baseline_value is not None, "You still need to compute baseline_value from past"
print("Baseline prediction:", baseline_value)
save("baseline", Baseline(baseline_value), [])

Baseline prediction: 0.27261592846745686
baseline: 0.5053 on past


## Part 4: Just One Feature (10 points)

Pick the one feature your Phase 3 plots say matters the most, and fit the simple model for your track using only that feature. Track A uses `LinearRegression` and Track B uses `LogisticRegression`.

**In the code cell:** set `FEATURE` to one column name from your `FEATURES` list. That's the only thing you change. The plot at the bottom is optional.

**Your answer:**
1. Which feature did you pick, and which figure made you pick it?
    * We picked `Critic_Score_Sq` because Figure A4 in our Phase 3 analysis showed that the titles with a higher critic score tend to have higher NA_Sales. We used the squared version instead of the raw score because Figure M2 showed a threshold: median sales stay nearly flat until a score of about 8 and then rise sharply, and squaring the score gives the model a column that also grows faster at high scores.
2. Did it beat the baseline on `past`? By how much?
    * Yes, this feature beat the baseline on `past`: it scored 0.4837 compared with the baseline's 0.5053, an improvement which is about a 0.022M (22,000 units per game), or about 4.27% lower error.

In [6]:
# Part 4: just one feature.
FEATURE = "Critic_Score_Sq"   # the one feature your plots say matters most

if TRACK == "A":
    model = LinearRegression()
else:
    model = LogisticRegression(max_iter=1000)

assert FEATURE in FEATURES, "Pick FEATURE from your FEATURES list in Part 1"

model.fit(scaled(past, [FEATURE]), past[TARGET])
save("one feature", model, [FEATURE], scale=True)

# Optional: plot your predictions against the real values, past only.

one feature: 0.4837 on past


## Part 5: Now Add All Your Features (10 points)

Same model, but now give it everything in `FEATURES`, including the columns you made in Part 1. The code also fits it once without your Part 1 columns so you can see if they actually helped.

**In the code cell:** nothing to fill in, just run it. If every one of your features is from Part 1, there's nothing to compare against, so just say that for question 2.

**Your answer:**
1. Did the score on `past` get better than Part 4? By how much?
    * Yes, the error on `past` dropped from 0.4837 with one feature to 0.4540 with all features, an improvement of about 0.03M (30,000 units per game), or about 6.14% lower error.

2. Did your Part 1 columns help? Compare "all features" and "without Part 1 columns" in the output.
    * Yes, adding our Part 1 columns lowered the error from 0.4789 to 0.4540, about 0.025M (25,000 units per game). Because `NEW_COLUMNS` includes the genre and platform dummies, this compares critic scores and release year alone against those features plus genre, platform, `Critic_Score_Sq` and `X360_Shooter`. Our extra "genre + platform only" model, which tests our research question directly, scored 0.4906, which is better than the baseline (0.5053), but worse than every model that includes critic score and release year.

3. Remember from lecture, what always happens to the score on the data the model learned from when you add more features? So does that make you trust this improvement more or less?
    * Adding more features will always make the error on the training data the same or lower, never worse, since the model can just ignore a column that doesn't help. So we trust this improvement less, because some of it might come from the model fitting random noise in `past` instead of real patterns, and we won't really know until we test it on `future`.

In [7]:
# Part 5: now all your features.
if TRACK == "A":
    model = LinearRegression()
else:
    model = LogisticRegression(max_iter=1000)

model.fit(scaled(past, FEATURES), past[TARGET])
save("all features", model, FEATURES, scale=True)

# Same model without your Part 1 columns so you can compare. Don't change this part.
OLD_COLUMNS = [c for c in FEATURES if c not in NEW_COLUMNS]
if OLD_COLUMNS:
    model_old = LinearRegression() if TRACK == "A" else LogisticRegression(max_iter=1000)
    model_old.fit(scaled(past, OLD_COLUMNS), past[TARGET])
    save("without Part 1 columns", model_old, OLD_COLUMNS, scale=True)

all features: 0.4540 on past
without Part 1 columns: 0.4789 on past


In [8]:
# Part 5 (extra): our two named predictors only, to answer the research question directly.
GP_COLUMNS = genre_cols + platform_cols + ["X360_Shooter"]

model_gp = LinearRegression()
model_gp.fit(scaled(past, GP_COLUMNS), past[TARGET])
save("genre + platform only", model_gp, GP_COLUMNS, scale=True)

genre + platform only: 0.4906 on past


## Part 6: Turn the Dial (15 points)

Every model has some kind of dial that controls how tightly it hugs the data it learned from. You're going to try three settings and see what happens.

* **Track A** uses Ridge or Lasso, and the dial is λ (it's called `alpha` in the code). A bigger λ means a stronger penalty, so the coefficients get pushed toward zero. Try three values that are really far apart, like 1, 1000, and 100000. If the scores barely change, go bigger, big datasets need a big λ before anything happens.
* **Track B** uses a decision tree, and the dial is `max_depth`, which is basically how many yes or no questions the tree gets to ask. Try a shallow one, a medium one, and one with no limit (`None`).

**In the code cell:** Track A, pick `Ridge` or `Lasso` on the marked line (Ridge is already there, swap it for Lasso if you want), and change the λ values only if the scores barely move. Track B, you don't have to change anything, but you can try different depths. The code saves all three for you.

Save all three and don't pick a winner yet. You'll find out who actually wins in Part 8.

**Your answer:**
1. As you turned the dial, what happened to the score on `past`?
    * As we increased $\lambda$ the error on `past` increased from 0.4541 $\rightarrow$ 0.4645 $\rightarrow$ 0.4959, meaning the model got worse on `past` as $\lambda$ grew. For $\lambda = 100000$ the model is almost as bad as the baseline (0.5053). This behavior makes sense because as $\lambda$ grows every coefficient gets pushed towards 0, and a model with all zero weights predicts the mean.

2. Track A, what happened to the coefficients as λ got bigger? If you used Lasso, did any go to exactly zero, and which ones? Track B, did any depth get close to 100% on `past`?
    * We used Ridge, as explained in Question 1 as $\lambda$ grows every coefficient gets pushed towards 0. For example, `Critic_Score` moved from -1.768 $\rightarrow$ 0.04 $\rightarrow$ 0.009, and `Critic_Score_Sq` moved from 1.386 $\rightarrow$ 0.163 $\rightarrow$ 0.011. `Critic_Score` starts negative and `Critic_Score_Sq` starts positive. These two columns are nearly copies of each other, so they cancel out. As $\lambda$ grows, Ridge removes that cancellation and `Critic_Score` flips to a positive sign.

3. Make a guess, which setting do you think will do best on `future`, and why?
    * We think setting $\lambda = 1$ will do best on `future`, because we have trained our model on `past` which is 80% of the sample and $\lambda = 1$ gave us the lowest error on `past`. When we test the remaining rows from `future`, the error should not change by a large number.

In [9]:
# Part 6: turn the dial. Try three settings and save all of them.
if TRACK == "A":
    SETTINGS = [1, 1000, 100000]        # λ values, keep them far apart. You can change these.
    coefs = {}
    for lam in SETTINGS:
        model = Ridge(alpha=lam) # or Lasso(alpha=lam). Pick one and tell us which.
        model.fit(scaled(past, FEATURES), past[TARGET])
        save(f"{type(model).__name__} λ={lam}", model, FEATURES, scale=True)
        coefs[f"λ={lam}"] = model.coef_
    display(pd.DataFrame(coefs, index=FEATURES).round(3))
else:
    SETTINGS = [2, 5, None]             # tree depths. None means no limit at all.
    for depth in SETTINGS:
        model = DecisionTreeClassifier(max_depth=depth, random_state=SEED)
        model.fit(past[FEATURES], past[TARGET])
        save(f"tree depth={depth or 'unlimited'}", model, FEATURES)

Ridge λ=1: 0.4541 on past
Ridge λ=1000: 0.4645 on past
Ridge λ=100000: 0.4959 on past


,λ=1,λ=1000,λ=100000
Critic_Score,-1.768,0.040,0.009
Has_Critic_Score,0.550,-0.072,0.007
Release_Year,-0.085,-0.053,-0.002
Critic_Score_Sq,1.386,0.163,0.011
X360_Shooter,0.024,0.023,0.005
...,...,...,...
Platform_PSV,-0.001,-0.005,-0.002
Platform_Wii,0.012,0.003,-0.000
Platform_X360,0.040,0.029,0.005
Platform_XB,-0.027,-0.020,-0.001


## Part 7: Random Forest (10 points)

A random forest grows a bunch of slightly different trees and lets them vote. Fit one for your track, Track A uses `RandomForestRegressor` and Track B uses `RandomForestClassifier`. The code also shows you which features the forest leaned on the most.

**In the code cell:** nothing to fill in, just run it. It might take a little longer than the other cells, that's normal.

**Your answer:**
1. Which three features did the forest lean on the most? Do they match the figures you picked in Part 1 and Part 4?
    * The forest leaned most on `Release_Year` (0.227), `Critic_Score_Sq` (0.214), and `Critic_Score` (0.211). The two critic score columns match the figures we used in Part 4: Figure A4 showed higher-scored titles tend to sell more, and Figure M2 showed the threshold at a score of about 8. `Release_Year` matches the plateau in Figure B4, which a forest can follow but a straight line cannot. Genre and platform ranked low (the best, `Genre_Shooter`, is only 0.043) because each 0/1 column gives the forest just one place to split.

2. How does its score on `past` compare to everything else so far? Before we open the envelope, how much do you actually trust that score?
    * The forest's error on `past` was 0.2873M, which is far lower than every other model so far (all between 0.45M and 0.51M). We still do not trust it much. Although `future` was drawn at random from the same data, so its rows should look a lot like `past`, the forest could have memorized individual titles from `past`, so its error could increase once it sees new rows.

In [10]:
# Part 7: random forest.
if TRACK == "A":
    model = RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1)
else:
    model = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)

model.fit(past[FEATURES], past[TARGET])
save("random forest", model, FEATURES)

importance = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
# display(importance.head(10).round(3))
# Switch to print due to display() not rendering the top 10 features.
print(importance.head(10).round(3))

random forest: 0.2873 on past
Release_Year       0.227
Critic_Score_Sq    0.214
Critic_Score       0.211
Genre_Shooter      0.043
Genre_Sports       0.032
X360_Shooter       0.029
Platform_X360      0.026
Platform_PS2       0.022
Platform_PS3       0.021
Platform_PS4       0.018
dtype: float64


## Part 8: Open the Envelope (10 points)

Okay, now we finally use `future`. Run the cell and it checks every model you saved on rows it has never seen. You get a table with each model's score on `past`, its score on `future`, and how much it changed.

**In the code cell:** nothing to fill in, just run it after Parts 0 to 7 are done.

How to read the change: on Track A a positive change means the error went up, so the model did worse on new rows. On Track B a negative change means accuracy went down, so again the model did worse.

**Your answer:**
1. Which model looked best on `past`? Which one is best on `future`? Is it the same model?
    * `past`: Random Forest, error = 0.2873.
    * `future`: Ridge λ = 1, error = 0.4319.
    * No, they are not the same model.

2. Which model changed the most between `past` and `future`? Think back to what we said about overfitting and explain why in 2 or 3 sentences.
    * The random forest model changed the most, with a change of 0.1789, while every other model changed only by about -0.02 to -0.03; the forest is the only model that did worse on the new rows. This is overfitting, because as its trees grew deeper the forest likely memorized the individual titles, including their noise and outliers, so its error jumped from 0.2873 on `past` to 0.4662 on `future`. This also meets the switch condition from our Phase 3 model case, where we said we would switch if a linear model on log(NA_Sales) reached about the same error as the forest model; here a linear model on raw NA_Sales actually beat the forest on `future` (0.4319 vs 0.4662).

3. Was your guess from Part 6 right?
    * Surprisingly, yes our guess from Part 6 was correct: λ = 1 at 0.4319 performed the best on `future`, beating λ = 1000 at 0.4391 and λ = 100000 at 0.4691.

4. **Your verdict.** In 3 to 5 sentences, answer your research question using your best `future` score. Then tell us one thing this model can't tell you, like a cause, a missing variable, or a group your data doesn't cover.
    * Genre and platform can predict North American video game sales, but only weakly, and those two predictors alone do not give the best predictions. Using only genre, platform, and `X360_Shooter`, our model scored 0.4663 on `future` compared with the baseline's 0.4794, only about a 2.73% lower error. On the other hand, utilizing critic score and release year produced our best model, Ridge with λ = 1, at 0.4319 on `future`, about a 9.9% lower error, which suggests that these two columns are better predictors for NA_Sales. However, this model which utilizes `Release_Year` does not account for the majority of titles on older platforms as they only have their biggest hits tracked in this dataset, so it may overestimate sales for older games.

In [11]:
# Part 8: open the envelope. Only run this once you're done with Parts 0 to 7.
results = open_envelope()
display(results)

# Optional: plot past and future scores side by side for each model.

,model,past,future,change
0,baseline,0.5053,0.4794,-0.0259
1,one feature,0.4837,0.4559,-0.0278
2,all features,0.4540,0.4321,-0.0219
3,without Part 1 columns,0.4789,0.4512,-0.0277
4,genre + platform only,0.4906,0.4663,-0.0243
5,Ridge λ=1,0.4541,0.4319,-0.0222
6,Ridge λ=1000,0.4645,0.4391,-0.0254
7,Ridge λ=100000,0.4959,0.4691,-0.0268
8,random forest,0.2873,0.4662,0.1789


## Part 9: README, Contributions, and AI Use (5 points)

Almost done. Add these:

1. A README listing the files you're submitting and what order to run things in.
2. Two or three sentences per member on what each of you did.
3. Which AI tools you used, if any, and what for. The Phase 2 AI Ethics Clearance still applies to any data you paste into an AI tool. Parts 0D, 6 question 3, and 8 need to be in your own words.
4. Run Kernel, Restart and Run All. If the code cell below prints its message, you're good.

**In the code cell:** nothing to fill in, it just confirms everything ran.

**Your answer:**
   * README:
       * Files Submitted:
           * Group6_Aaron_Sergio_Phase4.ipynb
           * Group6_Aaron_Sergio_Phase4.html
           * phase4_sample.csv
       * How to run: Place notebook and phase4_sample.csv in the same folder, then run Kernel $\rightarrow$ Restart and Run All. The notebook will run top to bottom, and the last cell confirms the run was successful.

   * Contributions:
      * Aaron: Set up the modeling data, drew the sample from the Phase 3 working dataset, and documented what had and hadn't changed since Phase 3. Turned Phase 3 figures into model features, including genre and platform dummy columns, the squared critic score, the X360 Shooter combination, and the critic-score flag that kept unreviewed titles in the data. Computed the baseline that every model had to beat.
      * Sergio: Built and compared the models: a one-feature linear regression on squared critic score, a model using all the features, and an extra genre and platform only model (the idea was Aaron's) to test the research question directly. Tuned the Ridge regression model across three penalty strengths, interpreted how coefficients shrank, and fit the random forest and read its feature importance.
      * Both: Wrote and proofread the answer portion of every section to include the final results and verdict.

   * AI use: We used Claude (Anthropic) to explain the differences between Ridge and Lasso model, and to fix the grammar of some long run-on sentences.

In [12]:
# Part 9: if this prints after Restart and Run All, you're good.
print(f"Ran top to bottom. Track {TRACK}, target {TARGET}, {len(scoreboard)} models saved.")

Ran top to bottom. Track A, target NA_Sales, 9 models saved.


## Grading Rubric (100 points)

Here's exactly what we're looking for in each part.

| Part | What we check | Points |
|---|---|---|
| **0** | **0A:** you filled in the track sentence and the reason matches your target type | 3 |
| | **0B:** all three change questions answered, any change explained, old and new question side by side, and permission noted if you switched datasets | 4 |
| | **0C:** full dataset link with shape and size, a sample of at most 20,000 rows made once with seed 577, and only `shape`, `dtypes`, and `sample(10)` shown | 4 |
| | **0D:** if something changed, catch-up plots plus your leakage and linear or forest call. If nothing changed, a real reason that points to your Phase 3 figures. "It was fine" gets 0 | 4 |
| **1** | Three new columns (a curve, a category, a combination) that only use their own row and never the target | 6 |
| | Each new column tied to a figure in one sentence | 2 |
| | `FEATURES` and `NEW_COLUMNS` run and everything is numeric | 2 |
| **2** | `past` and `future` made with the given code, and both shapes print | 3 |
| | Your answer on why unseen rows give a more honest score | 2 |
| **3** | Baseline computed from `past` with pandas (the mean for Track A, the most common class for Track B) and saved | 5 |
| | Baseline guess given in your target's units or class name, and its score explained in plain words | 5 |
| **4** | One feature picked, with a Phase 3 figure as the reason | 4 |
| | Right simple model for your track, fit on `past` and saved | 3 |
| | Compared to the baseline, in plain words with numbers | 3 |
| **5** | Model fit on `past` with all your features and saved | 3 |
| | Improvement over Part 4 given with numbers | 2 |
| | You used the comparison output to say whether your Part 1 columns helped | 2 |
| | You connected more features to a better score on learned data, and said what that means for trusting it | 3 |
| **6** | Three dial settings fit on `past` and saved (Track A says Ridge or Lasso, Track B uses three depths) | 6 |
| | Coefficient table shown (Track A) or all three depth scores shown (Track B) | 3 |
| | Questions 1 and 2 answered with specifics from your output | 4 |
| | Question 3 has a real guess with a reason, made before Part 8 | 2 |
| **7** | Random forest for your track fit on `past` and saved | 4 |
| | Feature importance shown and compared to your figures from Parts 1 and 4 | 3 |
| | Your answer on how much you trust the forest's `past` score | 3 |
| **8** | Results table shown | 2 |
| | Questions 1 to 3 answered with numbers from the table, including an overfitting explanation for the biggest change | 4 |
| | Verdict answers your research question with your best `future` score and names one thing the model can't tell you | 4 |
| **9** | README with your files and run order | 1 |
| | Contributions for each member | 1 |
| | AI use statement | 1 |
| | Runs top to bottom with Restart and Run All on `phase4_sample.csv`, and the last cell prints its message | 2 |
| | **Total** | **100** |

**Things that cost you points**

| What happened | Points off |
|---|---|
| Printing your whole DataFrame, or any output over 20 rows | 2 each time, up to 6 |
| Plotting, describing, or scoring `future` before Part 8 | 5 |
| Picking your Part 6 winner using `future` before Part 8 | 3 |
| A feature that leaks the target (built from it, or recorded after the outcome) | 5 |
| Notebook doesn't run top to bottom | The parts that don't run get 0 |
| Deleting parts or cells, or writing answers outside **Your answer** | 2 |